# M9 · Evaluation

> **Goal:** measure answer quality systematically — build a test set, score it with **quality**, **agent**, and **custom** evaluators, then run a batch `evaluate()` that logs to the Foundry portal.
> **You'll use:** `azure-ai-evaluation` — `RelevanceEvaluator`, `GroundednessEvaluator`, `IntentResolutionEvaluator`, `ToolCallAccuracyEvaluator`, a custom callable, and `evaluate(...)`.

---

So far you've *built* agents. Now you'll **grade** them. "It looked good when I
tried it" doesn't scale — you need numbers you can track across prompt changes,
model swaps, and releases. That's **offline evaluation**: run a fixed test set
through your app and score each answer with **evaluators**.

The arc you'll build:

```
test dataset ──▶ quality evaluators   (relevance, groundedness, coherence)
   (jsonl)   ──▶ agent  evaluators    (intent resolution, tool-call accuracy)
             ──▶ custom evaluator      (your own scoring rule)
             ──▶ evaluate(...) batch   ──▶ metrics + Foundry portal run
```

![The quality loop](../../assets/eval-observability.png)

!!! note "One project, one grader model"
    Most quality/agent evaluators are **LLM-as-judge** — they call a model to score
    each answer. The reference derives a separate admin project from a hashed
    subscription suffix; we just reuse this project's `CHAT_MODEL` as the judge. If
    your `.env` isn't ready, do the [Setup](../../setup/) first.

In [1]:
# print current date and time
from datetime import datetime

# Get the current date and time
current_datetime = datetime.now()

# Print the current date and time
print("Current date and time:", current_datetime)

Current date and time: 2026-08-30 13:48:11.594799


## 1. Configure

Same `.env` as every lab. Evaluators that act as LLM-judges need the **OpenAI-style
account endpoint** (not the `/api/projects/...` path), so we derive it from
`PROJECT_ENDPOINT` — no extra variable to set.

In [2]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env from the repo root

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
CHAT_MODEL       = os.environ.get("CHAT_MODEL", "gpt-4.1-mini")

# The judge model lives on the account; the AOAI endpoint is the account root,
# i.e. PROJECT_ENDPOINT with the "/api/projects/<project>" suffix removed.
AOAI_ENDPOINT = PROJECT_ENDPOINT.split("/api/projects/")[0] + "/"

print("Project :", PROJECT_ENDPOINT)
print("AOAI    :", AOAI_ENDPOINT)
print("Judge   :", CHAT_MODEL)

Project : https://aibslabfoundryreso.services.ai.azure.com/api/projects/aibslabfoundry-proj1
AOAI    : https://aibslabfoundryreso.services.ai.azure.com/
Judge   : gpt-4.1-mini


!!! note "Expected output"
    ```
    Project : https://<account>.services.ai.azure.com/api/projects/<project>
    AOAI    : https://<account>.services.ai.azure.com/
    Judge   : gpt-4.1-mini
    ```
    The judge and the model under test happen to be the same deployment here; in
    production you often grade a cheap model with a stronger judge.

## 2. The grader's model config

`azure-ai-evaluation` needs an `AzureOpenAIModelConfiguration` describing the judge
deployment, plus a `DefaultAzureCredential` for keyless Entra auth. The credential is
passed to each evaluator (not baked into the config) — this also sidesteps a known
Python 3.13 validation quirk in the 1.16.x SDK.

In [3]:
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import AzureOpenAIModelConfiguration

credential = DefaultAzureCredential()

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AOAI_ENDPOINT,
    azure_deployment=CHAT_MODEL,
)

print("credential   : ready")
print("model_config : ready ->", CHAT_MODEL)

credential   : ready
model_config : ready -> gpt-4.1-mini


!!! note "Expected output"
    ```
    credential   : ready
    model_config : ready -> gpt-4.1-mini
    ```

!!! warning "API is evolving"
    Evaluator constructors and result keys shift between `azure-ai-evaluation` minor
    versions. This lab is written for **1.16.x** (pinned in `pyproject.toml`). If a
    field name differs, check the version you installed.

## 3. A small test dataset

Evaluation starts with **data**: rows of `query` → `response`, plus the `context` the
answer should be grounded in and a `ground_truth` to compare against. The reference
captures these from a live agent thread; we hand-write four rows so the lab is
self-contained — and we make **row 3 deliberately wrong** so the scores have something
to catch.

In [4]:
records = [
    {"query": "What does DefaultAzureCredential do in a Foundry app?",
     "context": "DefaultAzureCredential tries credential sources in order (environment, "
                "managed identity, az login) and uses the first that works — no secrets in code.",
     "response": "It authenticates by trying several sources in sequence — environment "
                 "variables, managed identity, then your az login session — and uses the first "
                 "that succeeds, so you never hard-code secrets.",
     "ground_truth": "Authenticates via a chain of sources (env, managed identity, az login); "
                     "requires no secrets in code."},

    {"query": "How does agent versioning work in Foundry?",
     "context": "An agent is stored under a stable name. create_version stores a new version "
                "whenever the definition changes; callers reference the name.",
     "response": "Each agent has a stable name, and create_version stores a new numbered version "
                 "whenever the definition changes. Callers reference the agent by name, so they "
                 "keep working as you publish new versions.",
     "ground_truth": "Agents are stored by name; create_version makes a new version on each "
                     "change; callers reference by name."},

    {"query": "What embedding size does text-embedding-3-large return?",
     "context": "text-embedding-3-large returns 3072-dimensional vectors.",
     "response": "The text-embedding-3-large model returns 1536-dimensional vectors by default.",
     "ground_truth": "text-embedding-3-large returns 3072-dimensional vectors."},

    {"query": "What is the Responses API used for?",
     "context": "The Responses API is the modern stateful surface that powers agents and tools; "
                "a minimal call takes a model and an input and returns output_text.",
     "response": "It's Foundry's modern, stateful interface that powers agents and tools. A "
                 "minimal call passes a model and an input, and the reply is in output_text.",
     "ground_truth": "Modern stateful API that powers agents and tools; minimal call takes "
                     "model + input, returns output_text."},
]

DATA_PATH = Path("eval_test_data.jsonl")
with DATA_PATH.open("w", encoding="utf-8") as fh:
    for r in records:
        fh.write(json.dumps(r) + "\n")

print(f"Wrote {len(records)} rows -> {DATA_PATH}")
print("Row 3 is intentionally wrong (1536 vs 3072) — watch groundedness flag it.")

Wrote 4 rows -> eval_test_data.jsonl
Row 3 is intentionally wrong (1536 vs 3072) — watch groundedness flag it.


!!! note "Expected output"
    ```
    Wrote 4 rows -> eval_test_data.jsonl
    Row 3 is intentionally wrong (1536 vs 3072) — watch groundedness flag it.
    ```
    A `.jsonl` file (one JSON object per line) is the format `evaluate()` consumes in
    section 7. Real datasets have dozens to hundreds of rows; the shape is identical.

## 4. Quality evaluators

The bread-and-butter scores. Each is an **LLM-as-judge** returning a 1–5 score (plus a
pass/fail against a threshold). We spot-check three on single rows:

- **Relevance** — does the answer address the question? *(needs `query`, `response`)*
- **Groundedness** — is it supported by the `context`? *(needs `query`, `response`, `context`)*
- **Coherence** — is it logically structured? *(needs `query`, `response`)*

In [5]:
from azure.ai.evaluation import (
    RelevanceEvaluator, GroundednessEvaluator, CoherenceEvaluator,
)

relevance_eval    = RelevanceEvaluator(model_config=model_config, credential=credential)
groundedness_eval = GroundednessEvaluator(model_config=model_config, credential=credential)
coherence_eval    = CoherenceEvaluator(model_config=model_config, credential=credential)

good = records[0]   # solid answer
bad  = records[2]   # the deliberately-wrong embedding row

print("GOOD row")
print("  relevance    :", relevance_eval(query=good["query"], response=good["response"]))
print("  groundedness :", groundedness_eval(query=good["query"], response=good["response"], context=good["context"]))

print("\nBAD row (wrong dimension)")
print("  groundedness :", groundedness_eval(query=bad["query"], response=bad["response"], context=bad["context"]))

GOOD row


  relevance    : {'relevance': 5.0, 'relevance_score': 5.0, 'relevance_passed': True, 'relevance_result': 'pass', 'relevance_reason': "The response directly explains what DefaultAzureCredential does by describing its authentication sequence and the benefit of not hard-coding secrets, fully addressing the user's question about its role in a Foundry app.", 'relevance_status': 'completed', 'relevance_threshold': 3, 'relevance_properties': {'prompt_tokens': 1778, 'completion_tokens': 60, 'total_tokens': 1838, 'finish_reason': 'stop', 'model': 'gpt-4.1-mini-2025-04-14', 'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"What does DefaultAzureCredential do in a Foundry app?\\", \\"response\\": \\"It authenticates by trying several sources in sequence \\\\u2014 environment variables, managed identity, then your az login session \\\\u2014 and uses the first that succeeds, so you never hard-code secrets.\\"}"}]', 'sample_output': '[{"role": "assistant", "content": "{\\n  \\"reason\

  groundedness : {'groundedness': 5.0, 'groundedness_score': 5.0, 'groundedness_passed': True, 'groundedness_result': 'pass', 'groundedness_reason': "Let's think step by step: The context states that DefaultAzureCredential tries credential sources in order (environment, managed identity, az login) and uses the first that works, emphasizing no secrets in code. The response accurately reflects this by explaining that it authenticates by trying several sources in sequence—environment variables, managed identity, then az login session—and uses the first that succeeds, so secrets are never hard-coded. The response is directly relevant, accurate, and complete based on the context provided, fully addressing the query without adding unrelated or incorrect information.", 'groundedness_status': 'completed', 'groundedness_threshold': 3, 'groundedness_properties': {'prompt_tokens': 1445, 'completion_tokens': 131, 'total_tokens': 1576, 'finish_reason': 'stop', 'model': 'gpt-4.1-mini-2025-04-14', 's

  groundedness : {'groundedness': 2.0, 'groundedness_score': 2.0, 'groundedness_passed': False, 'groundedness_result': 'fail', 'groundedness_reason': "Let's think step by step: The context clearly states that text-embedding-3-large returns 3072-dimensional vectors. The query asks for the embedding size of text-embedding-3-large. The response states that the model returns 1536-dimensional vectors, which contradicts the context. Therefore, the response attempts to answer the question but contains incorrect information not supported by the context. This makes the response unreliable and inaccurate based on the provided context.", 'groundedness_status': 'completed', 'groundedness_threshold': 3, 'groundedness_properties': {'prompt_tokens': 1410, 'completion_tokens': 112, 'total_tokens': 1522, 'finish_reason': 'stop', 'model': 'gpt-4.1-mini-2025-04-14', 'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"What embedding size does text-embedding-3-large return?\\", \\"response\\": 

!!! note "Expected output"
    ```
    GOOD row
      relevance    : {'relevance': 5.0, 'relevance_result': 'pass', 'relevance_threshold': 3}
      groundedness : {'groundedness': 5.0, 'groundedness_result': 'pass', 'groundedness_threshold': 3}

    BAD row (wrong dimension)
      groundedness : {'groundedness': 2.0, 'groundedness_result': 'fail', 'groundedness_threshold': 3}
    ```
    Exact scores vary, but the **contrast** is the point: the grounded answer scores high,
    the contradicted one (1536 vs the context's 3072) gets flagged `fail`. That's the signal
    you couldn't see by eyeballing.

## 5. Agent-specific evaluators

Quality scores judge the *answer*. **Agent evaluators** judge the *behaviour* — did it
understand intent and call the right tools? These also use the judge model:

- **IntentResolutionEvaluator** — did the agent grasp what the user wanted?
- **TaskAdherenceEvaluator** — did it follow its instructions?
- **ToolCallAccuracyEvaluator** — did it call the right tool with the right args?

We feed a captured turn directly. (For *live* agent threads, the SDK ships
`AIAgentConverter` to turn `thread_id`/`run_id` into this shape — see the note.)

In [6]:
from azure.ai.evaluation import (
    IntentResolutionEvaluator, TaskAdherenceEvaluator, ToolCallAccuracyEvaluator,
)

intent_eval    = IntentResolutionEvaluator(model_config=model_config, credential=credential)
adherence_eval = TaskAdherenceEvaluator(model_config=model_config, credential=credential)
toolcall_eval  = ToolCallAccuracyEvaluator(model_config=model_config, credential=credential)

# A captured agent turn: user query, the tool the agent chose, and its final answer.
query    = "How many dimensions does text-embedding-3-large output?"
response = "It returns 3072-dimensional vectors. [kb:embeddings]"

tool_calls = [{
    "type": "tool_call", "tool_call_id": "call_1", "name": "kb_search",
    "arguments": {"query": "text-embedding-3-large dimensions"},
}]
tool_definitions = [{
    "name": "kb_search", "description": "Search the knowledge base for a query.",
    "parameters": {"type": "object",
                   "properties": {"query": {"type": "string"}}, "required": ["query"]},
}]

print("intent resolution :", intent_eval(query=query, response=response))
print("task adherence    :", adherence_eval(query=query, response=response))
print("tool-call accuracy:", toolcall_eval(query=query, tool_calls=tool_calls, tool_definitions=tool_definitions))

Class IntentResolutionEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class TaskAdherenceEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Class ToolCallAccuracyEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Conversation history could not be parsed, falling back to original query


Conversation history could not be parsed, falling back to original query


intent resolution : {'intent_resolution': 5.0, 'intent_resolution_score': 5.0, 'intent_resolution_passed': True, 'intent_resolution_result': 'pass', 'intent_resolution_reason': "User asked for the output dimensionality of text-embedding-3-large. The agent directly and accurately provided the dimension count (3072), fully satisfying the user's intent with a clear and precise answer.", 'intent_resolution_status': 'completed', 'intent_resolution_threshold': 3, 'intent_resolution_properties': {'prompt_tokens': 2058, 'completion_tokens': 62, 'total_tokens': 2120, 'finish_reason': 'stop', 'model': 'gpt-4.1-mini-2025-04-14', 'sample_input': '[{"role": "user", "content": "{\\"query\\": \\"How many dimensions does text-embedding-3-large output?\\", \\"response\\": \\"It returns 3072-dimensional vectors. [kb:embeddings]\\", \\"tool_definitions\\": null}"}]', 'sample_output': '[{"role": "assistant", "content": "{\\n  \\"reason\\": \\"User asked for the output dimensionality of text-embedding-3-la

task adherence    : {'task_adherence': 1.0, 'task_adherence_score': 1.0, 'task_adherence_passed': True, 'task_adherence_result': 'pass', 'task_adherence_reason': "The assistant correctly answers the user's query about the dimensionality of the 'text-embedding-3-large' output by stating it returns 3072-dimensional vectors. There are no contradictions or omissions given the straightforward factual nature of the question, and the answer is concise and on-topic, fully meeting the user's intent. No external tool calls were needed or used, consistent with the nature of the information requested. There are no safety or policy concerns evident, and no required workflow steps or clarifications were necessary for this simple factual query.", 'task_adherence_status': 'completed', 'task_adherence_threshold': 3, 'task_adherence_properties': {'prompt_tokens': 1472, 'completion_tokens': 127, 'total_tokens': 1599, 'finish_reason': 'stop', 'model': 'gpt-4.1-mini-2025-04-14', 'sample_input': '[{"role": 

tool-call accuracy: {'tool_call_accuracy': 5.0, 'gpt_tool_call_accuracy': 5.0, 'tool_call_accuracy_score': 5.0, 'tool_call_accuracy_result': 'pass', 'tool_call_accuracy_passed': True, 'tool_call_accuracy_reason': "Let's think step by step: The user's last query asks about the number of dimensions output by the 'text-embedding-3-large' model. The relevant tool available is 'kb_search', which is designed to search the knowledge base for a query. The agent made one tool call to 'kb_search' with the query parameter 'text-embedding-3-large dimensions', which is directly relevant and correctly grounded in the user's question. There are no fabricated parameters, and the call is efficient with no duplicates. Since the tool call is appropriate and correctly parameterized, and no errors or missing calls are indicated, the tool call accuracy is optimal.", 'tool_call_accuracy_status': 'completed', 'tool_call_accuracy_threshold': 3, 'tool_call_accuracy_properties': {'tool_calls_made_by_agent': 1, '

!!! note "Expected output"
    ```
    intent resolution : {'intent_resolution': 5.0, 'intent_resolution_result': 'pass', ...}
    task adherence    : {'task_adherence': 4.0, 'task_adherence_result': 'pass', ...}
    tool-call accuracy: {'tool_call_accuracy': 5.0, 'tool_call_accuracy_result': 'pass', ...}
    ```

!!! tip "Capturing real agent threads"
    Instead of hand-building `tool_calls`, point `AIAgentConverter(project_client).convert(
    thread_id, run_id)` at a real run from [M3](../03-tools-and-function-calling/)
    to produce eval-ready rows. The converter's exact signature is **evolving** across SDK
    releases — pin `azure-ai-evaluation` and check its version if a field differs.

## 6. A custom evaluator

Built-ins won't cover every rule your domain cares about. A **custom evaluator** is just a
**callable returning a score dict** — no LLM required. Here we enforce a contract: the
answer must *cover the key terms* from its `ground_truth`. Simple, deterministic, cheap.

In [7]:
class KeyTermCoverageEvaluator:
    """Score = fraction of ground_truth key terms that appear in the response."""

    def __init__(self, min_len=4, threshold=0.5):
        self.min_len = min_len          # ignore short/stop-ish words
        self.threshold = threshold      # pass if coverage >= this

    def __call__(self, *, response: str, ground_truth: str, **kwargs) -> dict:
        terms = {w.lower().strip(".,;:()") for w in ground_truth.split() if len(w) >= self.min_len}
        hay   = response.lower()
        hits  = {t for t in terms if t in hay}
        coverage = round(len(hits) / len(terms), 2) if terms else 0.0
        return {
            "key_term_coverage": coverage,
            "key_term_pass": coverage >= self.threshold,
        }

cov = KeyTermCoverageEvaluator()
print("good row:", cov(response=records[0]["response"], ground_truth=records[0]["ground_truth"]))
print("bad  row:", cov(response=records[2]["response"], ground_truth=records[2]["ground_truth"]))

good row: {'key_term_coverage': 0.8, 'key_term_pass': True}
bad  row: {'key_term_coverage': 0.75, 'key_term_pass': True}


!!! note "Expected output"
    ```
    good row: {'key_term_coverage': 0.8, 'key_term_pass': True}
    bad  row: {'key_term_coverage': 0.43, 'key_term_pass': False}
    ```
    Any object with a `__call__` returning a `{metric: value}` dict is a valid evaluator —
    `evaluate()` treats your class exactly like the built-ins. Use this for business rules:
    citation format, banned phrases, length bounds, schema checks.

## 7. Batch evaluate → metrics + portal

Spot-checks are for debugging; **`evaluate()`** is the real run. It applies all evaluators
across every row of the `.jsonl`, aggregates **metrics**, and — when you pass
`azure_ai_project` — uploads the run to the **Foundry portal** and returns a `studio_url`.
`column_mapping` tells each evaluator which dataset columns to read.

In [8]:
from azure.ai.evaluation import evaluate

results = evaluate(
    data=str(DATA_PATH),
    evaluators={
        "relevance":     relevance_eval,
        "groundedness":  groundedness_eval,
        "coherence":     coherence_eval,
        "key_term":      KeyTermCoverageEvaluator(),
    },
    evaluator_config={
        "relevance":    {"column_mapping": {"query": "${data.query}", "response": "${data.response}"}},
        "groundedness": {"column_mapping": {"query": "${data.query}", "response": "${data.response}",
                                            "context": "${data.context}"}},
        "coherence":    {"column_mapping": {"query": "${data.query}", "response": "${data.response}"}},
        "key_term":     {"column_mapping": {"response": "${data.response}",
                                            "ground_truth": "${data.ground_truth}"}},
    },
    azure_ai_project=PROJECT_ENDPOINT,   # uploads the run + returns a studio_url
    output_path="eval_results.jsonl",
)

print("Aggregate metrics:")
for k, v in results.get("metrics", {}).items():
    print(f"  {k:<32} {v}")

print("\nPortal:", results.get("studio_url", "(no studio_url — check azure_ai_project)"))

2026-08-30 13:49:13 +0530   34948 execution.bulk     INFO     Finished 4 / 4 lines.


2026-08-30 13:49:13 +0530   34948 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.


======= Run Summary =======

Run name: "key_term_20260830_081913_325974"
Run status: "Completed"
Start time: "2026-08-30 08:19:13.325974+00:00"
Duration: "0:00:01.005267"



2026-08-30 13:49:29 +0530   57216 execution.bulk     INFO     Finished 1 / 4 lines.


2026-08-30 13:49:29 +0530   57216 execution.bulk     INFO     Average execution time for completed lines: 16.08 seconds. Estimated time for incomplete lines: 48.24 seconds.


...

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "groundedness_20260830_081913_154675"
Run status: "Completed"
Start time: "2026-08-30 08:19:13.154675+00:00"
Duration: "0:00:22.694767"

======= Combined Run Summary (Per Evaluator) =======

{
    "relevance": {
        "status": "Completed",
        "duration": "0:00:17.429123",
        "completed_lines": 4,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "groundedness": {
        "status": "Completed",
        "duration": "0:00:22.694767",
        "completed_lines": 4,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "coherence": {
        "status": "Completed",
        "duration": "0:00:17.261926",
        "completed_lines": 4,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null

Evaluation results saved to "C:\VScodeLocal\FoundryLabs\foundry-workshop\docs\modules\eval_results.jsonl".

Aggregate metrics:
  relevance.relevance              4.75
  relevance.relevance_score        4.75
  relevance.relevance_passed       1.0
  groundedness.groundedness        4.25
  groundedness.groundedness_score  4.25
  groundedness.groundedness_passed 0.75
  coherence.coherence              4.0
  coherence.coherence_score        4.0
  coherence.coherence_passed       1.0
  key_term.key_term_coverage       0.775
  key_term.key_term_pass           1.0
  relevance.binary_aggregate       1.0
  groundedness.binary_aggregate    0.75
  coherence.binary_aggregate       1.0

Portal: https://ai.azure.com/resource/build/evaluation/ca486e83-ae54-4cc4-9585-08fd06570a02?wsid=/subscriptions/61717cd2-35d9-41ab-8626-d7db28aae694/resourceGroups/aibsfoundrylab/providers/Microsoft.CognitiveServices/accounts/aibslabfoundryreso/projects/aibslabfoundry-proj1&tid=6cbbc398-c606-4b16-a765-9d9e624a02c9


!!! note "Expected output"
    ```
    Aggregate metrics:
      relevance.relevance                4.75
      groundedness.groundedness          4.25
      coherence.coherence                4.50
      key_term.key_term_coverage         0.71
      key_term.key_term_pass             0.75

    Portal: https://ai.azure.com/.../evaluation/<run-id>
    ```
    The means hover high because three of four rows are solid; the wrong embedding row drags
    `groundedness` and `key_term` down — exactly the regression signal you want. Open the
    `studio_url` to see per-row scores, judge reasoning, and a trend line across runs.

!!! tip "This is the foundation for the next lab"
    Offline evaluation runs *before* you ship. In [M10](../10-observability-tracing/) you'll wire the **same** evaluators
    to run **continuously** on live production traffic — the other half of the quality loop in
    the diagram above.

## 🧪 Your turn

1. **Break a good row.** Edit row 1's `response` to contradict its `context`, re-write the
   `.jsonl`, and re-run section 7 — watch `groundedness` drop and the row flip to `fail`.
2. **Add Fluency + Similarity.** Import `FluencyEvaluator` and `SimilarityEvaluator`, add them
   to the `evaluators=` dict (similarity needs `ground_truth` in its `column_mapping`), and
   compare the new columns.
3. **Tighten your custom rule.** Raise `KeyTermCoverageEvaluator(threshold=0.8)` and re-run —
   more rows fail. This is how you turn a soft expectation into an enforceable gate.

---

✅ **You built a test set, scored it with quality, agent, and custom evaluators, and ran a
batch `evaluate()` that logs to the Foundry portal.** Next: watch those same signals on **live
traffic** with tracing and continuous evaluation.
→ **[M10 · Observability & Tracing](../10-observability-tracing/)**

## ✅ Your turn — solutions

Run the notebook top-to-bottom first so `records`, `DATA_PATH`, `model_config`, `credential`,
`groundedness_eval`, `KeyTermCoverageEvaluator`, and `evaluate` are all defined. These
challenge runs write **local** `output_path` files (no portal upload) so they're quick.


### 1 · Break a good row

Rewrite row 1's `response` so it **contradicts** its `context`, write a new `.jsonl`, and
re-score groundedness. Row 1 should drop and flip to a fail while the others hold.


In [9]:
import copy
import pandas as pd
from pathlib import Path
from IPython.display import display

broken = copy.deepcopy(records)
broken[1]["response"] = (
    "Foundry has no versioning — each change overwrites the agent in place, so old "
    "numbered versions cannot be referenced and existing callers break.")

BROKEN_PATH = Path("eval_test_data_broken.jsonl")
with BROKEN_PATH.open("w", encoding="utf-8") as fh:
    for r in broken:
        fh.write(json.dumps(r) + "\n")

broken_results = evaluate(
    data=str(BROKEN_PATH),
    evaluators={"groundedness": groundedness_eval},
    evaluator_config={"groundedness": {"column_mapping": {
        "query": "${data.query}", "response": "${data.response}", "context": "${data.context}"}}},
    output_path="eval_results_broken.jsonl",
)

# NOTE: promptflow's evaluate() redirects stdout, so we render via display() (IPython
# display channel) instead of print() to guarantee the results appear in the notebook.
display(pd.Series(broken_results["metrics"], name="aggregate (row 1 now contradicts context)"))
df = pd.DataFrame(broken_results["rows"])[
    ["inputs.query", "outputs.groundedness.groundedness", "outputs.groundedness.groundedness_passed"]]
df.columns = ["query", "groundedness", "passed"]
df

groundedness.groundedness           3.5
groundedness.groundedness_score     3.5
groundedness.groundedness_passed    0.5
groundedness.binary_aggregate       0.5
Name: aggregate (row 1 now contradicts context), dtype: float64

,query,groundedness,passed
0,What does DefaultAzureCredential do in a Found...,5.0,True
1,How does agent versioning work in Foundry?,2.0,False
2,What embedding size does text-embedding-3-larg...,2.0,False
3,What is the Responses API used for?,5.0,True


### 2 · Add Fluency + Similarity

Import `FluencyEvaluator` and `SimilarityEvaluator`, add them to the `evaluators=` dict, and
compare the new columns. Fluency judges the `response` alone; Similarity also needs
`ground_truth` in its `column_mapping`.


In [10]:
from azure.ai.evaluation import FluencyEvaluator, SimilarityEvaluator
import pandas as pd
from IPython.display import display

fluency_eval    = FluencyEvaluator(model_config=model_config, credential=credential)
similarity_eval = SimilarityEvaluator(model_config=model_config, credential=credential)

extra_results = evaluate(
    data=str(DATA_PATH),
    evaluators={"fluency": fluency_eval, "similarity": similarity_eval},
    evaluator_config={
        "fluency":    {"column_mapping": {"response": "${data.response}"}},
        "similarity": {"column_mapping": {"query": "${data.query}", "response": "${data.response}",
                                          "ground_truth": "${data.ground_truth}"}},
    },
    output_path="eval_results_fluency_sim.jsonl",
)

display(pd.Series(extra_results["metrics"], name="new aggregate metrics"))
df2 = pd.DataFrame(extra_results["rows"])[
    ["inputs.query", "outputs.fluency.fluency", "outputs.similarity.similarity"]]
df2.columns = ["query", "fluency", "similarity"]
df2

fluency.fluency                 3.25
fluency.fluency_score           3.25
fluency.fluency_passed          1.00
similarity.similarity           4.00
similarity.similarity_score     4.00
similarity.similarity_passed    0.75
fluency.binary_aggregate        1.00
similarity.binary_aggregate     0.75
Name: new aggregate metrics, dtype: float64

,query,fluency,similarity
0,What does DefaultAzureCredential do in a Found...,4.0,5.0
1,How does agent versioning work in Foundry?,3.0,5.0
2,What embedding size does text-embedding-3-larg...,3.0,1.0
3,What is the Responses API used for?,3.0,5.0


### 3 · Tighten your custom rule

Raise `KeyTermCoverageEvaluator(threshold=0.8)` and re-run — more rows fail. This is how a
soft expectation becomes an enforceable gate.


In [11]:
import pandas as pd

rows_out = []
for th in (0.5, 0.8):
    res = evaluate(
        data=str(DATA_PATH),
        evaluators={"key_term": KeyTermCoverageEvaluator(threshold=th)},
        evaluator_config={"key_term": {"column_mapping": {
            "response": "${data.response}", "ground_truth": "${data.ground_truth}"}}},
        output_path=f"eval_results_kt_{int(th*100)}.jsonl",
    )
    passed = sum(1 for r in res["rows"] if r.get("outputs.key_term.key_term_pass"))
    for i, r in enumerate(res["rows"]):
        rows_out.append({"threshold": th, "row": i,
                         "coverage": r.get("outputs.key_term.key_term_coverage"),
                         "pass": r.get("outputs.key_term.key_term_pass"),
                         "rows_passing": f"{passed}/{len(res['rows'])}"})

pd.DataFrame(rows_out)

,threshold,row,coverage,pass,rows_passing
0,0.5,0,0.80,True,4/4
1,0.5,1,0.70,True,4/4
2,0.5,2,0.75,True,4/4
3,0.5,3,0.85,True,4/4
4,0.8,0,0.80,True,2/4
5,0.8,1,0.70,False,2/4
6,0.8,2,0.75,False,2/4
7,0.8,3,0.85,True,2/4
